This notebook trains 2 machine learning models.

The preprocessing of the transcription data includes:
- Run the spaCy pipeline that includes **named entity recognition**
- Carry out lemmatization and remove clinically irrelevant stop words 

The ML pipeline is structured like this:
- Split the data into a training and testing dataset.
- Have a cross validation of a linear SVC pipeline and a multinomial logistic regression
- Determine the model with the best test matrix and fit it on the training dataset.

In [1]:
# Import the relevant libraries for preprocessing
import pandas as pd
import spacy

In [2]:
# Load the appropriate data
top_mtsamples = pd.read_csv("../data/top_mtsamples.csv")
top_mtsamples.head()

,description,medical_specialty,sample_name,transcription,keywords,medical_specialty_clean
0,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple...",Cardiovascular / Pulmonary
1,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo...",Cardiovascular / Pulmonary
2,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 3,"2-D ECHOCARDIOGRAM,Multiple views of the heart...","cardiovascular / pulmonary, 2-d echocardiogram...",Cardiovascular / Pulmonary
3,Echocardiogram and Doppler,Cardiovascular / Pulmonary,2-D Echocardiogram - 4,"DESCRIPTION:,1. Normal cardiac chambers size....","cardiovascular / pulmonary, ejection fraction,...",Cardiovascular / Pulmonary
4,"Normal left ventricle, moderate biatrial enla...",Cardiovascular / Pulmonary,2-D Doppler,"2-D STUDY,1. Mild aortic stenosis, widely calc...","cardiovascular / pulmonary, 2-d study, doppler...",Cardiovascular / Pulmonary


In [3]:
# Load the spaCy model
# nlp = spacy.load("en_core_sci_sm") # The clinical dataset is not available for download
nlp = spacy.load("en_core_web_sm")

# Load the clinical stopwords
with open('../data/clinical-stopwords.txt', 'r') as file:
    custom_stopwords = set(file.read().splitlines())

# Create a function that carries out the preprocessing
def extract_and_lemmatize_entities(text):
    # Handle missing/NaN transcripts gracefully
    if not isinstance(text, str):
        return ""
        
    doc = nlp(text)
    processed_entities = []
    
    # Iterate ONLY through the recognized named entities
    for ent in doc.ents:
        # Lemmatize and filter out custom stopwords and punctuation
        ent_tokens = [
            token.lemma_.lower() for token in ent 
            if token.lemma_.lower() not in custom_stopwords 
            and not token.is_punct
            and not token.is_space
        ]
        
        # If tokens remain after filtering, join them back into a string
        if ent_tokens:
            processed_entities.append(" ".join(ent_tokens))
            
    # Return a single string of all processed entities for TF-IDF
    return " ".join(processed_entities)

In [4]:
# Compute the entities using the above function
top_mtsamples['entity_features'] = top_mtsamples['transcription'].apply(extract_and_lemmatize_entities)

In [5]:
# Saving the data with entities to avoid rerunning the processing
top_mtsamples.to_pickle("../.private/top_mtsamples_preprocessed.pkl")

In [13]:
# Load the data that is preprocessed
top_mtsamples = pd.read_pickle("../.private/top_mtsamples_preprocessed.pkl")

## Carry out the machine learning pipeline

This pipeline carries out the linearSVC and multinomial logistic regression while creating a cross validation 5 times

In [14]:
# Load the required libraries for this section
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


In [15]:
# Split the data into a test and split section
# Define Features (X) and Target (y)
X = top_mtsamples['entity_features']
y = top_mtsamples['medical_specialty_clean']

# Train/Test Split (30% test size, random_state 42, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30,
    random_state=42, 
    stratify=y
)

In [16]:
# Define the pipeline 
# LinearSVC Pipeline
svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
                    ngram_range=(1, 2),      
                    min_df=3,                
                    max_df=0.9,               
                    max_features=20000,       
                    sublinear_tf=True,    
                    strip_accents='unicode',
                    norm='l2'
    )),
    ('clf', LinearSVC(class_weight='balanced', random_state=42))
])

# Multinomial Logistic Regression Pipeline
logreg_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),      
            min_df=3,                
            max_df=0.9,               
            max_features=20000,       
            sublinear_tf=True,    
            strip_accents='unicode',
            norm='l2'
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        random_state=42,
        max_iter=1000  # increased to help convergence with TF-IDF's high-dimensional sparse features
    ))
])

In [17]:
# Perform the cross validation on the training dataset
print("Running 5-fold CV for LinearSVC...")
svc_cv_results = cross_validate(
    svc_pipeline, X_train, y_train, 
    cv=5, 
    scoring=['accuracy', 'f1_macro']
)

print("Running 5-fold CV for Logistic Regression...")
logreg_cv_results = cross_validate(
    logreg_pipeline, X_train, y_train, 
    cv=5, 
    scoring=['accuracy', 'f1_macro']
)

Running 5-fold CV for LinearSVC...
Running 5-fold CV for Logistic Regression...


In [18]:
# Print training metrics
print("\n--- Cross-Validation Results ---")
print(f"LinearSVC Mean F1-Macro: {np.mean(svc_cv_results['test_f1_macro']):.4f}")
print(f"LogReg Mean F1-Macro:    {np.mean(logreg_cv_results['test_f1_macro']):.4f}")


--- Cross-Validation Results ---
LinearSVC Mean F1-Macro: 0.2995
LogReg Mean F1-Macro:    0.4186


In [19]:
# Fit the better pipeline on the full training set
logreg_pipeline.fit(X_train, y_train)

# Generate predictions on the unseen test set
y_pred = logreg_pipeline.predict(X_test)

# 3. Calculate the requested metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_f1_macro = f1_score(y_test, y_pred, average='macro')

# 4. Print the final results
print("\n--- Test Set Evaluation ---")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1-Macro: {test_f1_macro:.4f}")

print("\n--- Detailed Classification Report ---")
# The classification report provides precision, recall, and f1-score per class
print(classification_report(y_test, y_pred))


--- Test Set Evaluation ---
Accuracy: 0.4163
F1-Macro: 0.4150

--- Detailed Classification Report ---
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.40      0.39      0.39       111
          Gastroenterology       0.35      0.46      0.40        67
          General Medicine       0.48      0.73      0.58        78
                 Neurology       0.35      0.40      0.37        67
   Obstetrics / Gynecology       0.42      0.52      0.47        46
                Orthopedic       0.39      0.48      0.43       107
                 Radiology       0.21      0.24      0.23        82
                   Surgery       0.57      0.34      0.43       327
                   Urology       0.40      0.51      0.45        47

                  accuracy                           0.42       932
                 macro avg       0.40      0.45      0.41       932
              weighted avg       0.44      0.42      0.41       932

